In [ ]:
%cd ../src

In [ ]:
import logging
from datetime import datetime

import polars as pl
from dateutil.relativedelta import relativedelta

from brokers.kite import KiteHistorical, KiteLogin, fetch_instruments
from compute import cal_stocks_rs, gen_nse_base_population, run_scanner
from conf import KITE_CREDENTIALS_PATH
from config.brokers import KiteConfig
from config.compute import ScannerConfig
from config.file_paths import FilePathConfig
from ingest import fetch_nse_fno_stocks, fetch_nse_hist_data
from utils import setup_logger

logger = logging.getLogger()
setup_logger()

In [ ]:
END_DATE = datetime(2026, 11, 20, 0, 0, 0)
START_DATE = END_DATE - relativedelta(months=10)
end_date_str = END_DATE.strftime("%Y-%m-%d")
file_paths = FilePathConfig(end_date_str)

In [ ]:
END_DATE

In [ ]:
START_DATE

# Kite Login

In [ ]:
kite_conn = KiteLogin(credentials_path=KITE_CREDENTIALS_PATH).auto_login()

# Historical Data Download

In [ ]:
fetch_nse_hist_data(
    kite_conn=kite_conn,
    start_date=START_DATE.strftime("%Y-%m-%d %H:%M:%S"),
    end_date=END_DATE.strftime("%Y-%m-%d %H:%M:%S"),
    frequency="day",
    save_path=end_date_str,
)

In [ ]:
nse_fno_stock_data = fetch_nse_fno_stocks(kite_conn=kite_conn, save_path=end_date_str)

# Load Data

In [ ]:
_tbl_name = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency="day", failed_tbl=False
)

stock_ohlcv_data = pl.read_database_uri(
    query=f"SELECT * from {_tbl_name}", uri=KiteConfig.DB_CONN, engine="connectorx"
)

In [ ]:
base_population_data = gen_nse_base_population(data=stock_ohlcv_data)
base_population_data.write_csv(file_paths.base_population)

In [ ]:
long_data, short_data = run_scanner(
    base_pop_data=base_population_data,
    stock_ohlcv_data=stock_ohlcv_data,
    fno_stocks=nse_fno_stock_data,
    end_date=END_DATE,
    scan_conf=ScannerConfig,
)

In [ ]:
long_data.write_csv(file_paths.long_stocks)
short_data.write_csv(file_paths.short_stocks)

# Fetching Indices Data

In [ ]:
df = fetch_instruments(kite_conn, exchange="NSE")

In [ ]:
indices_df = df.filter(
    pl.col("symbol").is_in(["NIFTY MIDSML 400", "NIFTY SMLCAP 250"])
).select("symbol", "instrument_token")

In [ ]:
frequency = "day"
kite_hist = KiteHistorical(
    kite_conn,
    instruments_df=indices_df,
    config=KiteConfig,
)

_tbl_name = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_INDICES, frequency=frequency, failed_tbl=False
)
_fl_tbl_name = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_INDICES, frequency=frequency, failed_tbl=True
)

kite_hist.get_historical_data(
    start_date=START_DATE.strftime("%Y-%m-%d %H:%M:%S"),
    end_date=END_DATE.strftime("%Y-%m-%d %H:%M:%S"),
    frequency=frequency,
    db_conn=KiteConfig.DB_CONN,
    insert_table_name=_tbl_name,
    failed_table_name=_fl_tbl_name,
)

# Relative Streangth Calculation

In [ ]:
base_population_stocks_list = (
    pl.scan_csv(file_paths.base_population)
    .select("symbol")
    .collect()
    .get_column("symbol")
    .to_list()
)
scan_conf = ScannerConfig

In [ ]:
frequency = "day"

_ind_tbl_name = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_INDICES, frequency=frequency, failed_tbl=False
)


_stock_tbl_name = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency=frequency, failed_tbl=False
)


stock_ohlcv_data = pl.read_database_uri(
    query=f"SELECT * from {_stock_tbl_name}",
    uri=KiteConfig.DB_CONN,
    engine="connectorx",
).filter(pl.col("symbol").is_in(base_population_stocks_list))


indices_ohlcv_data = pl.read_database_uri(
    query=f"SELECT * from {_ind_tbl_name}", uri=KiteConfig.DB_CONN, engine="connectorx"
).filter(pl.col("symbol") == "NIFTY MIDSML 400")

In [ ]:
rs_df = cal_stocks_rs(
    indices_df=indices_ohlcv_data,
    stocks_df=stock_ohlcv_data,
    end_date=END_DATE,
    scan_conf=ScannerConfig,
)

In [ ]:
rs_df.write_csv(file_paths.relative_strength)